# 3. Análise de termos e bigramas dos títulos
Esta etapa identifica os termos e pares de termos mais frequentes nos títulos tratados. Os resultados servirão de base para selecionar as expressões utilizadas na classificação pelos eixos da BNCC.

In [ ]:
from pathlib import Path
import math
import sys
import pandas as pd
from nltk.corpus import stopwords

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import limpar_texto
pasta_processados = raiz / 'dados' / '1_processados'

## 3.1 Leitura dos títulos tratados

In [ ]:
df = pd.read_csv(pasta_processados / '02_artigos_pre_processados.csv', encoding='utf-8-sig')
print(f'Artigos recebidos: {len(df)}')
df[['id_artigo', 'evento', 'ano', 'titulo_limpo', 'texto_limpo']].head()

## 3.2 Remoção de stopwords
Serão removidas as stopwords em português disponibilizadas pelo NLTK e palavras com até dois caracteres.

In [ ]:
stopwords_pt = {limpar_texto(palavra) for palavra in stopwords.words('portuguese')}

# Tokeniza o título, removendo stopwords e palavras com menos de 3 caracteres
def tokenizar_titulo(titulo_limpo):
    return [
        palavra for palavra in str(titulo_limpo).split()
        if palavra not in stopwords_pt and len(palavra) > 2
    ]

# Aplica a tokenização ao título limpo e cria uma nova coluna com os tokens
df['tokens_titulo'] = df['titulo_limpo'].fillna('').apply(tokenizar_titulo)
df[['titulo_limpo', 'tokens_titulo']].head(10)

## 3.3 Termos dos títulos

In [ ]:
# Cria um DataFrame com os termos extraídos dos títulos, associando-os aos artigos correspondentes
termos = (df[['id_artigo', 'ano', 'evento', 'tokens_titulo']]
    .explode('tokens_titulo')
    .rename(columns={'tokens_titulo': 'termo'})
    .dropna(subset=['termo']))

# Cria um ranking de termos com base na frequência e na quantidade de artigos em que aparecem
ranking_termos = (termos.groupby('termo')
    .agg(frequencia=('termo', 'size'), quantidade_artigos=('id_artigo', 'nunique'))
    .reset_index()
    .sort_values(['frequencia', 'termo'], ascending=[False, True]))
ranking_termos.head(20)

## 3.4 Bigramas dos títulos

In [ ]:
# Cria bigramas a partir dos tokens do título
def criar_bigramas(tokens):
    return [' '.join(par) for par in zip(tokens, tokens[1:])]

# Aplica a criação de bigramas ao título tokenizado e cria uma nova coluna com os bigramas
df['bigramas_titulo'] = df['tokens_titulo'].apply(criar_bigramas)
bigramas = (df[['id_artigo', 'ano', 'evento', 'bigramas_titulo']]
    .explode('bigramas_titulo')
    .rename(columns={'bigramas_titulo': 'bigrama'})
    .dropna(subset=['bigrama']))

# Cria um ranking de bigramas com base na frequência e na quantidade de artigos em que aparecem
ranking_bigramas = (bigramas.groupby('bigrama')
    .agg(frequencia=('bigrama', 'size'), quantidade_artigos=('id_artigo', 'nunique'))
    .reset_index()
    .sort_values(['frequencia', 'bigrama'], ascending=[False, True]))
ranking_bigramas.head(20)

## 3.5 Critérios de relevância
Seguindo os testes iniciais do estudo, serão considerados relevantes os termos com frequência mínima equivalente a 2% do corpus e os bigramas com frequência mínima equivalente a 1%.

In [ ]:
limite_termos = math.ceil(len(df) * 0.02)
limite_bigramas = math.ceil(len(df) * 0.01)

# Filtra os termos e bigramas relevantes com base nos limites definidos
ranking_termos_relevantes = ranking_termos[ranking_termos['frequencia'] >= limite_termos].copy()
ranking_bigramas_relevantes = ranking_bigramas[ranking_bigramas['frequencia'] >= limite_bigramas].copy()

# Cria um DataFrame com as medidas e valores calculados
pd.DataFrame({
    'medida': ['Artigos', 'Limite termos', 'Limite bigramas', 'Termos relevantes', 'Bigramas relevantes'],
    'valor': [len(df), limite_termos, limite_bigramas, len(ranking_termos_relevantes), len(ranking_bigramas_relevantes)],
})

In [ ]:
ranking_termos_relevantes

In [ ]:
ranking_bigramas_relevantes

## 3.6 Frequência por ano e evento

In [ ]:
termos_relevantes = termos[termos['termo'].isin(ranking_termos_relevantes['termo'])].copy()
bigramas_relevantes = bigramas[bigramas['bigrama'].isin(ranking_bigramas_relevantes['bigrama'])].copy()

freq_termos_ano_evento = (termos_relevantes.groupby(['ano', 'evento', 'termo'])
    .agg(frequencia=('termo', 'size'), quantidade_artigos=('id_artigo', 'nunique'))
    .reset_index())
freq_bigramas_ano_evento = (bigramas_relevantes.groupby(['ano', 'evento', 'bigrama'])
    .agg(frequencia=('bigrama', 'size'), quantidade_artigos=('id_artigo', 'nunique'))
    .reset_index())
freq_termos_ano_evento.head()

## 3.7 Exportação para seleção dos descritores BNCC

In [ ]:
termos_relevantes.to_csv(pasta_processados / '03_termos_titulos.csv', index=False, encoding='utf-8-sig')
bigramas_relevantes.to_csv(pasta_processados / '03_bigramas_titulos.csv', index=False, encoding='utf-8-sig')
ranking_termos_relevantes.to_csv(pasta_processados / '03_ranking_termos_titulos.csv', index=False, encoding='utf-8-sig')
ranking_bigramas_relevantes.to_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', index=False, encoding='utf-8-sig')
freq_termos_ano_evento.to_csv(pasta_processados / '03_frequencia_termos_ano_evento.csv', index=False, encoding='utf-8-sig')
freq_bigramas_ano_evento.to_csv(pasta_processados / '03_frequencia_bigramas_ano_evento.csv', index=False, encoding='utf-8-sig')
print(f'Termos relevantes: {len(ranking_termos_relevantes)}')
print(f'Bigramas relevantes: {len(ranking_bigramas_relevantes)}')